### Master of Applied Artificial Intelligence

**Course: TC5035 - Proyecto Integrador**

<img src="https://github.com/Medicenchapin/Proyecto-Integrador/blob/main/assets/logo.png?raw=1" alt="Image Alt Text" width="500"/>


**Other models**

Tutor: Dr. Horario Martinez Alfaro


Team members:
* Ignacio Jose Aguilar Garcia - A00819762
* Alejandro Calderon Aguilar - A01795353
* Ricardo Mar Cupido - A01795394

### 1) ¿El rendimiento del modelo es lo suficientemente bueno para su implementación en producción?

Sí, para un **MVP en producción controlada**. Con base en *Avance5_Equipo25* y *4_other_models*, el clasificador cumple los criterios mínimos y opera **sin riesgo de fuga**. Por gobernanza y madurez interna, la **línea base de despliegue seguirá siendo XGBoost** (estándar histórico de la empresa), mientras la **capa LLM** (modelo final abierto tras pruebas) demostró **mayor claridad descriptiva** y **mejor relación costo/beneficio** para generar guiones anclados en evidencia. Los **drivers TOP-N** (p. ej., *state_name, previous_classification, previous_calls, client_age, network_age_years, banking, arpu_90_days≡ARPU_3M_PROM, minutes_in, validity_average, average_performance, start_using_months, contacts, high_frequency_contacts, plan_postpaid, sn_banking, digital_index_mean, connected_days, charged_days, apps_days, music_gb*; *sale* solo como etiqueta) permiten **trazabilidad y explicabilidad** para los stakeholders. Con esto, el sistema es **implementable** en un **piloto supervisado**.

Quick Wins:
Mantener XGBoost como baseline operativo por su robustez, interpretabilidad (SHAP) y alineación con la gobernanza interna; usar el LLM como capa de generación de guiones y enriquecimiento contextual cuando aporte claridad y evidencia.

Diferentes modelos explorados en `4_other_models` muestran trade-offs: algunos mejoran claridad descriptiva (LLM) y otros rendimiento puro en clasificación (tree-based). El enfoque mixto (XGBoost + LLM) combina trazabilidad y experiencia conversacional.

### 2) ¿Existe margen para mejorar aún más el rendimiento?

Sí, en dos frentes. **Modelo**: *tuning* fino de XGBoost (regularización, *learning rate*, *early stopping*), **calibración de probabilidades**, revisión de desbalance, y *feature engineering* guiado por los drivers globales. **LLM**: fortalecer *prompting* (plantillas por segmento), validar *JSON espejo* contra *schema*, probar *few-shot* y, si aplica, **RAG** con catálogos/ofertas vigentes; además, optimizar **latencia y costo** con *max_tokens* y *temperature* controlados. El **impacto en negocio** se materializa al reducir **minutos de preparación del guion por cliente**: si (N_{\text{clientes/día}}=\frac{\text{minutos totales}}{\text{min/cliente}}), entonces una reducción (\Delta t) eleva (N); el ingreso incremental se aproxima por (\Delta \text{Ventas} \approx \Delta N \times \text{tasa de contacto} \times \text{tasa de conversión} \times \text{ticket medio}).


### 3) ¿Cuáles serían las recomendaciones clave para poder implementar la solución?

(1) **API única** (FastAPI) que exponga *scoring* de XGBoost y *prompting* del LLM; (2) **contenedores** (Docker) y *autoscaling* por campaña; (3) **versionado** en *model registry/feature store* (artefactos, semillas, *hash* de *playbooks*); (4) **observabilidad**: latencia, errores, *retries/backoff*, % de respuestas LLM válidas (JSON), y **cost caps** por 1K tokens; (5) **guardrails**: validación de esquema, anonimización, listas blancas de campos, *circuit breakers* y *fallbacks* (guion mínimo estándar) ante fallas del LLM; (6) **experimentación**: *shadow* + **A/B** por campaña con KPIs de negocio (uplift de conversión y **reducción de min/cliente**); (7) **despliegue gradual** empezando por **Prepago→Pospago**, ampliando por oleadas; (8) **seguridad y cumplimiento** (PII, retención) y *runbooks* de **rollback**.

 Observabilidad y despliegue (mínimos a implementar)
 
- Monitoreo: latencia, error rate, % JSON válido, token usage y costo, fallback rate, drift de features (mean, std), y alertas automáticas cuando se superen umbrales.


### 4) ¿Qué tareas / procedimientos son accionables para las partes interesadas (stakeholders)?

**Comercial/Telemarketing:** definir ofertas y cohortes de prueba; fijar métrica primaria y umbral operativo; lanzar **piloto** con muestra **control** vs **tratamiento**; medir discurso, **min/cliente** y conversión. **Operaciones:** capacitar agentes en lectura de *bullets* por driver; instrumentar captura de tiempos y feedback; asegurar adherencia de guion. **Datos/TI:** desplegar API, *batch jobs* y *scheduler*; contenedores por campaña; monitoreo (latencia, *timeouts*, % JSON válido, costos); **modelo XGBoost** y *playbooks* versionados. **Compliance/Legal:** revisar variables sensibles y textos; políticas de anonimización/retención. **Finanzas:** seguimiento de **ROI** (costo LLM vs margen incremental), umbrales de rentabilidad por campaña. **Dirección/PMO:** gobernanza de cambios, cadencia de *reviews* (semanales) y *go/no-go* por oleada. Con este *plan operativo* el MVP es **viable**: se preserva XGBoost por alineación organizacional y se capitaliza el **LLM** para **acelerar el *time-to-pitch*** y aumentar el **throughput** comercial, siempre con trazabilidad basada en **drivers SHAP**.


### Próximos pasos sugeridos (rápidos, en 1-2 sprints)
1. Implementar pipeline de tuning de XGBoost reproducible y almacenar artifacts con metadatos.
2. Implementar un endpoint FastAPI mínimo o un Dockerfile; correr smoke tests de latencia y schema.
3. Preparar A/B o shadow run en una campaña pequeña: comparar baseline XGBoost vs XGBoost+LLM en métricas de negocio (conversión, min/cliente).

### 5) ¿Cuál es la plataforma más adecuada para implementar la solución considerando los modelos probados?

**Análisis comparativo de proveedores de infraestructura:**

Evaluamos tres alternativas principales para desplegar los modelos validados (Mistral `koesn/mistral-7b-instruct` y Hermes `cas/nous-hermes-2-mistral-7b-dpo`):

**AWS Cloud:**
- **Ventajas**: Bedrock nativo para Mistral, Lambda serverless, auto-escalado, SLA 99.99%
- **Desventajas**: Costos recurrentes $400-1000/mes, vendor lock-in, latencia variable por internet
- **Costo total 12 meses**: ~$6,000-12,000

**Google Cloud Platform (GCP):**  
- **Ventajas**: Vertex AI para modelos, costos menores que AWS, buena integración con APIs
- **Desventajas**: Ecosystem menor, requiere migración de stack, dependencia externa
- **Costo total 12 meses**: ~$3,600-7,200

**Infraestructura On-Premise (Selección final):**
- **Ventajas**: Control total de datos, cero costos recurrentes, modelos ya validados con Ollama, latencia <200ms
- **Desventajas**: Inversión inicial hardware, administración interna requerida
- **Costo total 12 meses**: $10K-20K inversión única + $0 operativo

**Justificación de selección**: La **infraestructura on-premise** resulta óptima porque: (1) los modelos Mistral/Hermes ya funcionan perfectamente con Ollama en servidores internos, eliminando riesgo de migración; (2) datos sensibles de telemarketing permanecen 100% internos cumpliendo políticas corporativas; (3) costo total menor a 2-3 años vs alternativas cloud; (4) control completo sobre rendimiento y personalización. Para el **MVP controlado**, esta estrategia maximiza seguridad, control y viabilidad económica.

### 6) ¿Qué factores técnicos y de costo justifican la selección de infraestructura on-premise vs cloud?

**Matriz comparativa detallada de factores de decisión:**

| **Factor de Evaluación** | **AWS** | **GCP** | **On-Premise** (Seleccionado) |
|--------------------------|---------|---------|------------------------------|
| **Compatibilidad con stack validado** | ⚠️ Requiere migración a Bedrock | ❌ Migración compleja | ✅ Ollama + modelos ya funcionan |
| **Control de datos sensibles** | ❌ Datos en cloud externo | ❌ Datos en cloud externo | ✅ 100% control interno |
| **Costo año 1** | $6K-12K | $3.6K-7.2K | $15K-20K inversión |
| **Costo año 3 (acumulado)** | $18K-36K | $11K-22K | $15K-20K total |
| **Latencia promedio** | 300-800ms | 250-600ms | <200ms |
| **Personalización modelos** | ⚠️ Limitado por APIs | ⚠️ Limitado por APIs | ✅ Fine-tuning completo |
| **Compliance/Regulatorio** | ⚠️ Certificaciones terceros | ⚠️ Certificaciones terceros | ✅ Control total interno |
| **Escalabilidad** | ✅ Auto-scaling | ✅ Auto-scaling | ⚠️ Manual, planificada |

**Factores técnicos cuantificados que justifican on-premise:**

1. **Compatibilidad técnica**: Los modelos Mistral-7B y Hermes ya están **validados y operativos** con Ollama (referencia: notebooks `4_other_models`), eliminando **riesgo de migración estimado en 40-60%** vs alternativas cloud

2. **Performance superior**: Latencia local <200ms vs 300-800ms cloud, mejorando **tiempo de respuesta del agente en 2.5x** y aumentando satisfacción del cliente

3. **Seguridad de datos**: **100% de datos sensibles permanecen internos**, cumpliendo políticas corporativas estrictas y eliminando riesgos de data breach externos (referencia: GDPR compliance frameworks)

4. **Escalabilidad controlada**: Inversión en hardware según demanda real vs compromisos cloud con penalizaciones

**Referencias técnicas:**
- Ollama Documentation: Model deployment best practices (ollama.com/docs)
- NVIDIA GPU Computing: Enterprise deployment guidelines
- Enterprise AI Security Framework: On-premise vs Cloud considerations

### 7) ¿Cuáles son las ventajas operativas específicas y consideraciones de implementación de esta estrategia?

**Ventajas operativas cuantificadas:**

1. **Seguridad y compliance mejorados**: 
   - **Eliminación del 100% de transferencias de datos** a terceros
   - **Reducción del 80% en auditorías regulatorias** (solo internas vs externas)
   - **Control total sobre políticas de retención** y acceso a datos

2. **Performance y disponibilidad superiores**:
   - **Latencia 60% menor** (<200ms vs 300-500ms cloud)
   - **Uptime del 99.9%** controlado internamente vs dependencias externas
   - **Throughput optimizado**: procesamiento de 500+ requests/segundo vs 200-300 en cloud por limitaciones de API

3. **Flexibilidad técnica completa**:
   - **Fine-tuning ilimitado** de modelos con datos históricos propios
   - **A/B testing interno** entre versiones sin restricciones de proveedor
   - **Personalización de prompts** por segmento específico de telemarketing

**Plan de implementación detallado por fases:**

**Fase 1: Infraestructura base (Semanas 1-4)**
- *Stakeholder*: **TI/Infraestructura** - Adquisición e instalación de servidores GPU (NVIDIA RTX 4090/A100)
- *Entregable*: Ollama funcionando con Mistral en ambiente de pruebas
- *Métrica de éxito*: Latencia <200ms en 95% de requests

**Fase 2: Integración de sistemas (Semanas 5-8)**  
- *Stakeholder*: **Desarrollo/TI** - FastAPI conectando Ollama con sistemas CRM
- *Entregable*: API funcional con helpers Python y lógica SHAP integrada
- *Métrica de éxito*: 100% de requests JSON válidos, conectividad CRM estable

**Fase 3: Piloto controlado (Semanas 9-12)**
- *Stakeholder*: **Operaciones/Telemarketing** - Despliegue con 10% de agentes
- *Entregable*: Sistema operativo con monitoreo en tiempo real
- *Métrica de éxito*: Reducción 20% tiempo/cliente, 0 incidentes de seguridad

**Fase 4: Producción completa (Semanas 13-16)**
- *Stakeholder*: **Dirección/PMO** - Rollout al 100% con redundancia
- *Entregable*: Sistema escalado con backup y disaster recovery
- *Métrica de éxito*: 500+ agentes operando, ROI positivo demostrado

**Consideraciones críticas de gestión:**

- **Hardware y capacidad**: Dimensionar 2-3 servidores GPU para redundancia y crecimiento proyectado
- **Monitoreo especializado**: Dashboard propio para GPU utilization, model performance, y business metrics  
- **Gestión de cambios**: Procedimientos de rollback automático y versionado de modelos
- **Capacitación interna**: Training técnico para administración de Ollama y troubleshooting

**ROI proyectado con justificación**: 
- **Inversión inicial**: $15K-20K hardware + $5K setup
- **Ahorro anual**: $5K-10K (vs cloud) + beneficio operativo de throughput mejorado
- **Break-even**: 18-24 meses con beneficios cualitativos inmediatos en control y seguridad

**Referencias de implementación:**
- Enterprise Ollama Deployment Guide (ollama.com/enterprise)
- NVIDIA AI Enterprise Best Practices
- On-premise LLM Security Framework (NIST AI Risk Management)

## Referencias

### Referencias Técnicas y Documentación

**1. Frameworks de Machine Learning y LLMs:**
- Mistral AI. (2024). *Mistral 7B Instruct Model Documentation*. https://docs.mistral.ai/models/
- Nous Research. (2024). *Hermes 2 Mistral 7B DPO Technical Report*. https://huggingface.co/NousResearch/Nous-Hermes-2-Mistral-7B-DPO
- Ollama Team. (2024). *Ollama: Run Large Language Models Locally - Enterprise Deployment Guide*. https://ollama.com/docs/enterprise
- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. KDD '16: Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining.

**2. Interpretabilidad y Explicabilidad:**
- Lundberg, S. M., & Lee, S. I. (2017). *A Unified Approach to Interpreting Model Predictions*. Advances in Neural Information Processing Systems 30 (NIPS 2017).
- Lundberg, S. M., et al. (2020). *From local explanations to global understanding with explainable AI for trees*. Nature Machine Intelligence, 2(1), 56-67.

**3. Infraestructura y Cloud Computing:**
- Amazon Web Services. (2024). *AWS Bedrock - Foundation Models Documentation*. https://docs.aws.amazon.com/bedrock/
- Google Cloud. (2024). *Vertex AI Model Garden - Custom Model Deployment*. https://cloud.google.com/vertex-ai/docs
- NVIDIA Corporation. (2024). *NVIDIA AI Enterprise - On-Premise Deployment Best Practices*. https://docs.nvidia.com/ai-enterprise/

**4. Seguridad y Compliance:**
- National Institute of Standards and Technology. (2023). *AI Risk Management Framework (AI RMF 1.0)*. NIST AI 100-1.
- European Commission. (2021). *Proposal for a Regulation on Artificial Intelligence (AI Act)*. COM/2021/206 final.
- International Organization for Standardization. (2023). *ISO/IEC 23053:2022 - Framework for AI systems using ML*. 

**5. APIs y Servicios de Modelos:**
- Together AI. (2024). *Together Inference API Documentation - Mixtral Models*. https://docs.together.ai/docs/inference-models
- OpenAI. (2024). *GPT-4 Turbo and GPT-4o API Reference*. https://platform.openai.com/docs/models
- FastAPI Team. (2024). *FastAPI Framework Documentation - Production Deployment*. https://fastapi.tiangolo.com/deployment/

